# CoralTeams - Remote LLM Server

Runs a local model on Colab's T4 GPU and exposes it via ngrok.
The semantic engine in your Codespace connects to this as an OpenAI-compatible API.

**No company data ever leaves your control** — Colab is used only for GPU compute.
The model and inference stay inside this notebook session.

In [ ]:
# @title 1. Install Ollama
import subprocess, time, sys, os, json, threading, urllib.request

print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

In [ ]:
# @title 2. Start Ollama Server
import os
# Start Ollama in background
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
!nohup ollama serve > /tmp/ollama.log 2>&1 &
print("Waiting for Ollama to start...")
!sleep 3
!curl -s http://localhost:11434/api/tags | head -c 200
print("\nOllama server running.")

In [ ]:
# @title 3. Pull Model
# Choose one:
MODEL = "qwen2.5:7b"       # Qwen 2.5 7B Instruct (recommended first)
# MODEL = "llama3.2:3b"    # Llama 3.2 3B (faster, lower quality)
# MODEL = "qwen2.5:3b"     # Qwen 2.5 3B (fastest, weakest)

print(f"Pulling {MODEL}...")
!ollama pull $MODEL
print(f"{MODEL} ready.")

In [ ]:
# @title 4. Quick Test
import json, urllib.request
body = json.dumps({
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a SQL generator. Answer concisely."},
        {"role": "user", "content": "Generate a SQL query: show open issues in mojombo/grit"}
    ]
}).encode()
req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=body,
    headers={"Content-Type": "application/json"}
)
resp = urllib.request.urlopen(req, timeout=120)
result = json.loads(resp.read())
print("Model response:")
print(result["choices"][0]["message"]["content"])
print("\n✓ Inference works.")

In [ ]:
# @title 5. Install ngrok and Expose the API
import urllib.request, zipfile, os, time

# Download ngrok
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -O /tmp/ngrok.tgz
!tar xzf /tmp/ngrok.tgz -C /usr/local/bin/

# Start ngrok tunnel on Ollama port
!nohup ngrok http 11434 --log=stdout > /tmp/ngrok.log 2>&1 &
print("Waiting for ngrok...")
time.sleep(3)

# Get the public URL
try:
    req = urllib.request.Request("http://localhost:4040/api/tunnels")
    resp = urllib.request.urlopen(req)
    data = json.loads(resp.read())
    public_url = data["tunnels"][0]["public_url"]
    print("=" * 60)
    print(f"NGROK URL: {public_url}")
    print("=" * 60)
    print()
    print("In your Codespace .env, set:")
    print(f"  MODEL_PROVIDER=openai")
    print(f"  MODEL_BASE_URL={public_url}/v1")
    print(f"  MODEL_NAME={MODEL}")
    print(f"  MODEL_API_KEY=")  # no auth needed for ngrok tunnel
    print()
    print("Then restart the semantic engine:")
    print("  pkill -f uvicorn")
    print("  cd /workspaces/Coral-Workspace/semantic-engine")
    print("  nohup uvicorn app.main:app --host 0.0.0.0 --port 8001 > /tmp/sem-engine.log 2>&1 &")
except Exception as e:
    print("ngrok URL fetch failed. Check /tmp/ngrok.log")
    print(str(e))

In [ ]:
# @title 6. Keep-Alive (prevents Colab from disconnecting)
import time, threading

def ping_ngrok():
    while True:
        try:
            urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
        except:
            pass
        time.sleep(60)

t = threading.Thread(target=ping_ngrok, daemon=True)
t.start()
print("Keep-alive thread running (pings ngrok every 60s).")
print()
print("Leave this Colab tab open. The notebook will keep running.")

In [ ]:
# @title 7. Test the Public Endpoint (run FROM your Codespace)
# Copy the URL from cell 5 and test it here on Colab first:
print("Testing public endpoint from Colab itself...")
import urllib.request
try:
    req = urllib.request.Request("http://localhost:4040/api/tunnels")
    resp = urllib.request.urlopen(req)
    data = json.loads(resp.read())
    public_url = data["tunnels"][0]["public_url"] + "/v1/chat/completions"
    
    body = json.dumps({
        "model": MODEL,
        "messages": [
            {"role": "user", "content": "Say hello and tell me what model you are"}
        ]
    }).encode()
    req2 = urllib.request.Request(public_url, data=body, headers={"Content-Type": "application/json"})
    resp2 = urllib.request.urlopen(req2, timeout=60)
    result = json.loads(resp2.read())
    print("Public endpoint response:")
    print(result["choices"][0]["message"]["content"])
    print("\n✓ Public endpoint works through ngrok.")
except Exception as e:
    print(f"Public test failed: {e}")